# In Class Activity April 14th, 2026

In [2]:
!pip install optuna

zsh:1: command not found: pip


### Importing libraries, preparing data, initial EDA

In [3]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# importing data
adult = pd.read_csv('../Data/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report_04_14.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data

adult.isnull().sum()


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report_04_14.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64

In [6]:
adult.dtypes


age                int64
workclass            str
fnlwgt             int64
education            str
educational-num    int64
marital-status       str
occupation           str
relationship         str
race                 str
gender               str
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country       str
income               str
dtype: object

### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





I learned that we have null values and also many categorical values along with numerical. However there seems to be a lot of categorical variables so I am thinking about potentually using catboost as a preliminary model. There is also class imbalance for our target variable so we need to take that into account

### Data Preprocessing (minimal) and Baseline Model

In [7]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

/var/folders/kk/czyhbh5961ggwtk14x632d900000gq/T/ipykernel_80624/2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [8]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [9]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The model has no tuning so that is something I will look into. Also I will use the scale pos weight to account for class imbalance.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [10]:
# your code here
scale_pos_weight = 1.0
max_depth = 10
learning_rate = 0.2
xgboost_model = XGBClassifier(enable_categorical=True, random_state=42, scale_pos_weight=scale_pos_weight, max_depth=max_depth, learning_rate=learning_rate)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgboost_model, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}')



Cross-validated F1 scores: [0.69981499 0.7103193  0.69055375 0.71348837 0.70242775]
Mean F1 score: 0.703320829286399


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [12]:
# your code here add verbose=1 to see the progress
param_grid = {
    'scale_pos_weight': [1.0, 2.0, 3.0, 4.0],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.2, 0.3],
    'n_estimators': [100, 200],
    'gamma': [0, 0.1, 0.2, 0.3]
  
}

xgb_grid = XGBClassifier(enable_categorical=True, random_state=42)
grid_search = GridSearchCV(xgb_grid, param_grid, cv=skf, scoring='f1', verbose=2)
grid_search.fit(X_train, y_train)

print(f'Best parameters from GridSearchCV: {grid_search.best_params_}')
print(f'Best F1 score from GridSearchCV: {grid_search.best_score_}')

xgb_grid_best = XGBClassifier(enable_categorical=True, random_state=42, **grid_search.best_params_)
xgb_grid_best.fit(X_train, y_train)










Fitting 5 folds for each of 288 candidates, totalling 1440 fits
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=1.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=1.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=1.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=1.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=1.0; total time=   0.2s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=2.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=2.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=2.0; total time=   0.1s
[CV] END gamma=0, learning_rate=0.1, max_depth=3, n_estimators=1

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [13]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': [1.0, 2.0, 3.0, 4.0],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.2, 0.3],
    'n_estimators': [100, 200],
    'gamma': [0, 0.1, 0.2, 0.3]
}

# replace this placeholder model with your preferred model from above

xgb_tuned = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_tuned.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_tuned.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_tuned.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_tuned_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_tuned.best_params_['scale_pos_weight'], 
                                max_depth=xgb_tuned.best_params_['max_depth'], 
                                learning_rate=xgb_tuned.best_params_['learning_rate'], 
                                n_estimators=xgb_tuned.best_params_['n_estimators'],
                                gamma=xgb_tuned.best_params_['gamma'],
                                enable_categorical=True)
xgb_tuned_best.fit(X_train, y_train)
y_pred_random = xgb_tuned_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': 2.0, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.3}
Best F1 score from RandomizedSearchCV: 0.7286579570879569
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.91      7431
           1       0.68      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [14]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    gamma = trial.suggest_float('gamma', 0, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 21:33:58,180] A new study created in memory with name: no-name-2cca06b6-232e-4573-8dac-b145c5fe492e
Best trial: 0. Best value: 0.615411:   5%|▌         | 1/20 [00:00<00:16,  1.16it/s]

[I 2026-04-15 21:33:59,057] Trial 0 finished with value: 0.615411310314466 and parameters: {'scale_pos_weight': 9.364235120434534, 'max_depth': 3, 'learning_rate': 0.06306185366406364, 'n_estimators': 106, 'gamma': 0.27827313067261916}. Best is trial 0 with value: 0.615411310314466.


Best trial: 1. Best value: 0.679033:  10%|█         | 2/20 [00:02<00:19,  1.09s/it]

[I 2026-04-15 21:34:00,301] Trial 1 finished with value: 0.6790326428676471 and parameters: {'scale_pos_weight': 7.871170685136163, 'max_depth': 6, 'learning_rate': 0.29962160805546606, 'n_estimators': 164, 'gamma': 0.0919380909698861}. Best is trial 1 with value: 0.6790326428676471.


Best trial: 2. Best value: 0.706506:  15%|█▌        | 3/20 [00:04<00:26,  1.57s/it]

...

[I 2026-04-15 21:34:21,767] Trial 17 finished with value: 0.706872765156122 and parameters: {'scale_pos_weight': 4.185051395907226, 'max_depth': 8, 'learning_rate': 0.2604684945228291, 'n_estimators': 184, 'gamma': 0.07266551681154702}. Best is trial 10 with value: 0.729966711549908.


Best trial: 10. Best value: 0.729967:  95%|█████████▌| 19/20 [00:24<00:01,  1.15s/it]

[I 2026-04-15 21:34:22,539] Trial 18 finished with value: 0.7234123297523167 and parameters: {'scale_pos_weight': 1.9326373605340885, 'max_depth': 3, 'learning_rate': 0.17322916165026353, 'n_estimators': 155, 'gamma': 0.21333330565526296}. Best is trial 10 with value: 0.729966711549908.


Best trial: 10. Best value: 0.729967: 100%|██████████| 20/20 [00:25<00:00,  1.27s/it]

[I 2026-04-15 21:34:23,643] Trial 19 finished with value: 0.6822144707141631 and parameters: {'scale_pos_weight': 6.264703283588426, 'max_depth': 5, 'learning_rate': 0.1979182951503735, 'n_estimators': 181, 'gamma': 0.2696900011937085}. Best is trial 10 with value: 0.729966711549908.
Best parameters from Optuna: {'scale_pos_weight': 1.4938115209630496, 'max_depth': 4, 'learning_rate': 0.2356521229322022, 'n_estimators': 198, 'gamma': 0.18855239532975565}
Best F1 score from Optuna: 0.729966711549908
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.91      0.92      7431
           1       0.73      0.74      0.73      2338

    accuracy                           0.87      9769
   macro avg       0.82      0.83      0.82      9769
weighted avg       0.87      0.87      0.87      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


With the different tuning methods Grid search took the longest by far. Then random was next followed by Optuna. Grid search had a better score then random but optuna had the best scores out of all of the models. The differences were pretty small however the timing difference between grid search and optuna was huge. Clearly optuna is the best option as it had the best scores and the shortest tuning time.